# EDA — biomarcadores acústicos de voz e doença de Parkinson

Este notebook descreve o **UCI 489 — Parkinson Dataset with replicated acoustic features**. O objetivo é examinar estrutura, cobertura, diversidade e associações exploratórias sem confundir 240 gravações com 240 pessoas.

> **Aviso de aquisição:** os dados não estão no Git. Com `DATA_SOURCE = "online"`, a célula de carregamento usa `fetch_ucirepo(id=489)` e requer internet. Com `DATA_SOURCE = "local"`, ela lê o CSV indicado em `LOCAL_DATA_PATH`. Nenhuma célula deste notebook foi executada durante a criação do repositório, para cumprir a decisão de não baixar o dataset nesta etapa.

Uso clínico potencial: suporte futuro a triagem não invasiva ou avaliação complementar. Esta análise não é diagnóstico médico e não valida uso clínico.

## 1. Objetivo e método

Pergunta de pesquisa: **características acústicas extraídas da voz apresentam diferenças mensuráveis entre indivíduos com doença de Parkinson e controles saudáveis e podem contribuir para diferenciar os dois grupos?**

Decisões metodológicas:

- `ID` é a unidade de agrupamento;
- as três gravações de uma pessoa são dependentes;
- a visão por gravação descreve o arquivo, mas as comparações principais usam a mediana por participante;
- `Status` é o alvo (0 = controle; 1 = Parkinson);
- análises e p-valores são exploratórios, com ajuste Benjamini–Hochberg quando várias características são comparadas;
- não há treinamento de classificador nesta entrega.

## 2. Preparação

### 2.1 Importações

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

### 2.2 Configuração visível e caminhos relativos

In [ ]:
# Troque para "local" se o CSV oficial já estiver em data/raw/.
DATA_SOURCE = "online"
LOCAL_DATA_PATH = Path("data/raw/ReplicatedAcousticFeatures-ParkinsonDatabase.csv")
AGGREGATION = "median"
RANDOM_STATE = 42

candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path.resolve() for path in candidate_roots if (path / "src" / "data_loader.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Raiz do projeto não encontrada. Execute o notebook a partir da raiz ou de notebooks/."
    )

LOCAL_DATA_PATH = PROJECT_ROOT / LOCAL_DATA_PATH
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_loader import load_parkinson_data

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Fonte selecionada: {DATA_SOURCE}")

### 2.3 Carregamento

Esta é a única célula que pode acessar a internet. Antes de executá-la, confirme `DATA_SOURCE`. O modo online usa a interface documentada pela UCI; o modo local nunca tenta baixar dados.

In [ ]:
data = load_parkinson_data(
    source=DATA_SOURCE,
    local_path=LOCAL_DATA_PATH if DATA_SOURCE == "local" else None,
)
print(f"Dados carregados: {data.shape[0]} linhas × {data.shape[1]} colunas")

## 3. Etapas da exploração

### 3.1 Inspeção estrutural

In [ ]:
display(data.head())
display(
    pd.DataFrame(
        {
            "coluna": data.columns,
            "tipo": data.dtypes.astype(str).values,
            "nao_nulos": data.notna().sum().values,
            "unicos": data.nunique(dropna=False).values,
        }
    )
)

### 3.2 Identificação dos campos documentados

In [ ]:
def get_column(frame: pd.DataFrame, expected: str) -> str:
    matches = {str(column).strip().casefold(): column for column in frame.columns}
    try:
        return matches[expected.casefold()]
    except KeyError as error:
        raise KeyError(f"Coluna {expected!r} não encontrada em {list(frame.columns)}") from error


ID_COLUMN = get_column(data, "ID")
RECORDING_COLUMN = get_column(data, "Recording")
STATUS_COLUMN = get_column(data, "Status")
GENDER_COLUMN = get_column(data, "Gender")

metadata_columns = [ID_COLUMN, RECORDING_COLUMN, STATUS_COLUMN, GENDER_COLUMN]
numeric_columns = data.select_dtypes(include="number").columns.tolist()
acoustic_columns = [column for column in numeric_columns if column not in metadata_columns]

print("Campos de contexto:", metadata_columns)
print("Número de características acústicas numéricas:", len(acoustic_columns))

### 3.3 Valores ausentes e duplicatas

In [ ]:
missing_summary = (
    data.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda table: 100 * table["valores_ausentes"] / len(data))
)
display(missing_summary.query("valores_ausentes > 0"))
print("Total de valores ausentes:", int(data.isna().sum().sum()))
print("Linhas totalmente duplicadas:", int(data.duplicated().sum()))

### 3.4 Distribuição das classes e número de indivíduos

In [ ]:
recording_class_distribution = (
    data[STATUS_COLUMN]
    .value_counts(dropna=False)
    .sort_index()
    .rename(index={0: "Controle", 1: "Parkinson"})
    .rename("n_gravacoes")
)
subject_class_distribution = (
    data.groupby(ID_COLUMN, observed=True)[STATUS_COLUMN]
    .first()
    .value_counts(dropna=False)
    .sort_index()
    .rename(index={0: "Controle", 1: "Parkinson"})
    .rename("n_participantes")
)

display(pd.concat([recording_class_distribution, subject_class_distribution], axis=1))
print("Participantes únicos:", data[ID_COLUMN].nunique())
print("Registros/gravações:", len(data))

### 3.5 Distribuição de sexo

In [ ]:
gender_by_subject = data.groupby(ID_COLUMN, observed=True)[GENDER_COLUMN].first()
gender_distribution = (
    gender_by_subject.value_counts(dropna=False)
    .sort_index()
    .rename(index={0: "Homem", 1: "Mulher"})
    .rename("n_participantes")
)
display(gender_distribution.to_frame())

gender_status = (
    data.groupby(ID_COLUMN, observed=True)[[GENDER_COLUMN, STATUS_COLUMN]]
    .first()
    .groupby([STATUS_COLUMN, GENDER_COLUMN], observed=True)
    .size()
    .unstack(fill_value=0)
    .rename(index={0: "Controle", 1: "Parkinson"}, columns={0: "Homem", 1: "Mulher"})
)
display(gender_status)

### 3.6 Gravações por indivíduo e consistência interna

In [ ]:
subject_structure = (
    data.groupby(ID_COLUMN, observed=True)
    .agg(
        n_gravacoes=(RECORDING_COLUMN, "size"),
        n_status=(STATUS_COLUMN, "nunique"),
        status=(STATUS_COLUMN, "first"),
        n_generos=(GENDER_COLUMN, "nunique"),
        genero=(GENDER_COLUMN, "first"),
    )
    .reset_index()
)

display(subject_structure.head(10))
display(subject_structure["n_gravacoes"].value_counts().sort_index().rename("n_participantes"))
print("Participantes com Status inconsistente:", int((subject_structure["n_status"] != 1).sum()))
print("Participantes com Gender inconsistente:", int((subject_structure["n_generos"] != 1).sum()))

### 3.7 Verificações de sanidade do desenho

In [ ]:
expected_status = {0, 1}
observed_status = set(pd.to_numeric(data[STATUS_COLUMN], errors="raise").unique())

assert data[ID_COLUMN].nunique() == 80, "A UCI documenta 80 participantes."
assert len(data) == 240, "A UCI documenta 240 gravações."
assert (subject_structure["n_gravacoes"] == 3).all(), "Nem todo ID possui três gravações."
assert (subject_structure["n_status"] == 1).all(), "Status varia dentro de pelo menos um ID."
assert observed_status == expected_status, f"Rótulos inesperados: {observed_status}"

print("Verificações do desenho replicado concluídas.")

### 3.8 Estatísticas descritivas por gravação

In [ ]:
recording_descriptive = data[acoustic_columns].describe().T
display(recording_descriptive)

scale_summary = recording_descriptive[["min", "max", "mean", "std"]].copy()
scale_summary["amplitude"] = scale_summary["max"] - scale_summary["min"]
display(scale_summary.sort_values("amplitude", ascending=False).head(15))

### 3.9 Representação agregada por participante

A mediana das três gravações reduz a influência de uma réplica extrema e produz uma linha por indivíduo. `Status` e `Gender` são carregados uma única vez por `ID`, depois de verificarmos sua consistência.

In [ ]:
if AGGREGATION not in {"median", "mean"}:
    raise ValueError("AGGREGATION deve ser 'median' ou 'mean'.")

grouped_acoustics = data.groupby(ID_COLUMN, observed=True)[acoustic_columns]
aggregated_acoustics = getattr(grouped_acoustics, AGGREGATION)()
subject_metadata = data.groupby(ID_COLUMN, observed=True)[[STATUS_COLUMN, GENDER_COLUMN]].first()
subject_data = subject_metadata.join(aggregated_acoustics).reset_index()

print(f"Tabela por participante ({AGGREGATION}): {subject_data.shape}")
display(subject_data.head())
display(subject_data[acoustic_columns].describe().T)

### 3.10 Famílias de características

In [ ]:
feature_families = {
    "jitter": ["Jitter_rel", "Jitter_abs", "Jitter_RAP", "Jitter_PPQ"],
    "shimmer": ["Shim_loc", "Shim_dB", "Shim_APQ3", "Shim_APQ5", "Shim_APQ11"],
    "hnr": ["HNR05", "HNR15", "HNR25", "HNR35", "HNR38"],
    "mfcc": [f"MFCC{i}" for i in range(13)],
    "delta_mfcc": [f"Delta{i}" for i in range(13)],
    "nao_lineares_ruido": ["RPDE", "DFA", "PPE", "GNE"],
}

available_families = {
    family: [column for column in columns if column in subject_data.columns]
    for family, columns in feature_families.items()
}
display(
    pd.DataFrame(
        {
            "familia": available_families.keys(),
            "n_variaveis": [len(columns) for columns in available_families.values()],
            "variaveis": [", ".join(columns) for columns in available_families.values()],
        }
    )
)

### 3.11 Comparação visual Parkinson versus controle

In [ ]:
preferred_features = ["Jitter_rel", "Shim_loc", "HNR05", "MFCC0", "RPDE", "DFA", "PPE", "GNE"]
representative_features = [feature for feature in preferred_features if feature in acoustic_columns][:4]
if not representative_features:
    representative_features = acoustic_columns[:4]

figure, axes = plt.subplots(
    len(representative_features),
    2,
    figsize=(12, 3.4 * len(representative_features)),
    squeeze=False,
)

for row, feature in enumerate(representative_features):
    for status, label, color in [(0, "Controle", "#4C78A8"), (1, "Parkinson", "#E45756")]:
        values = subject_data.loc[subject_data[STATUS_COLUMN] == status, feature].dropna()
        axes[row, 0].hist(values, bins="auto", alpha=0.55, label=label, color=color)
    axes[row, 0].set_title(f"Distribuição por participante — {feature}")
    axes[row, 0].set_xlabel(feature)
    axes[row, 0].set_ylabel("Participantes")
    axes[row, 0].legend()

    groups = [
        subject_data.loc[subject_data[STATUS_COLUMN] == status, feature].dropna()
        for status in (0, 1)
    ]
    axes[row, 1].boxplot(groups, tick_labels=["Controle", "Parkinson"], showfliers=True)
    axes[row, 1].set_title(f"Resumo por grupo — {feature}")
    axes[row, 1].set_ylabel(feature)

figure.tight_layout()
plt.show()

### 3.12 Correlações e possível redundância

In [ ]:
correlation_matrix = subject_data[acoustic_columns].corr(method="pearson")
upper_mask = np.triu(np.ones(correlation_matrix.shape, dtype=bool), k=1)
correlation_pairs = (
    correlation_matrix.where(upper_mask)
    .stack()
    .rename("correlacao")
    .reset_index()
    .rename(columns={"level_0": "variavel_1", "level_1": "variavel_2"})
)
correlation_pairs["correlacao_absoluta"] = correlation_pairs["correlacao"].abs()
display(correlation_pairs.sort_values("correlacao_absoluta", ascending=False).head(20))

In [ ]:
corr_features = [feature for feature in preferred_features if feature in acoustic_columns]
if len(corr_features) < 3:
    corr_features = acoustic_columns[: min(10, len(acoustic_columns))]

visual_correlation = subject_data[corr_features].corr(method="pearson")
figure, axis = plt.subplots(figsize=(9, 7))
image = axis.imshow(visual_correlation, vmin=-1, vmax=1, cmap="coolwarm")
axis.set_xticks(range(len(corr_features)), corr_features, rotation=45, ha="right")
axis.set_yticks(range(len(corr_features)), corr_features)
axis.set_title("Correlação de Pearson — subconjunto representativo por participante")
figure.colorbar(image, ax=axis, label="Correlação")
figure.tight_layout()
plt.show()

### 3.13 Associação exploratória com `Status`

São calculados, em nível de participante: correlação ponto-bisserial, diferença de médias, diferença de medianas, tamanho de efeito padronizado (Cohen's d) e Mann–Whitney. Os p-valores são ajustados por Benjamini–Hochberg. Isso organiza sinais para investigação; não confirma biomarcadores nem causalidade.

In [ ]:
def cohens_d(control: pd.Series, parkinson: pd.Series) -> float:
    control = control.dropna().astype(float)
    parkinson = parkinson.dropna().astype(float)
    degrees_freedom = len(control) + len(parkinson) - 2
    if degrees_freedom <= 0:
        return np.nan
    pooled_variance = (
        (len(control) - 1) * control.var(ddof=1)
        + (len(parkinson) - 1) * parkinson.var(ddof=1)
    ) / degrees_freedom
    if pooled_variance <= 0 or not np.isfinite(pooled_variance):
        return np.nan
    return (parkinson.mean() - control.mean()) / np.sqrt(pooled_variance)


def benjamini_hochberg(p_values: pd.Series) -> pd.Series:
    values = p_values.to_numpy(dtype=float)
    adjusted = np.full_like(values, np.nan)
    valid_positions = np.flatnonzero(np.isfinite(values))
    if len(valid_positions) == 0:
        return pd.Series(adjusted, index=p_values.index)
    valid_values = values[valid_positions]
    order = np.argsort(valid_values)
    ranked = valid_values[order]
    ranks = np.arange(1, len(ranked) + 1)
    ranked_adjusted = np.minimum.accumulate((ranked * len(ranked) / ranks)[::-1])[::-1]
    ranked_adjusted = np.clip(ranked_adjusted, 0, 1)
    restored = np.empty_like(ranked_adjusted)
    restored[order] = ranked_adjusted
    adjusted[valid_positions] = restored
    return pd.Series(adjusted, index=p_values.index)


association_rows = []
status = subject_data[STATUS_COLUMN].astype(int)
for feature in acoustic_columns:
    values = pd.to_numeric(subject_data[feature], errors="coerce")
    valid = values.notna() & status.notna()
    control = values[valid & (status == 0)]
    parkinson = values[valid & (status == 1)]

    if values[valid].nunique() < 2:
        point_biserial, point_p = np.nan, np.nan
    else:
        point_result = stats.pointbiserialr(status[valid], values[valid])
        point_biserial, point_p = point_result.statistic, point_result.pvalue

    if len(control) and len(parkinson):
        mann_result = stats.mannwhitneyu(parkinson, control, alternative="two-sided")
        mann_p = mann_result.pvalue
    else:
        mann_p = np.nan

    association_rows.append(
        {
            "variavel": feature,
            "media_controle": control.mean(),
            "media_parkinson": parkinson.mean(),
            "diferenca_medias": parkinson.mean() - control.mean(),
            "diferenca_medianas": parkinson.median() - control.median(),
            "cohens_d": cohens_d(control, parkinson),
            "r_ponto_bisserial": point_biserial,
            "p_ponto_bisserial": point_p,
            "p_mann_whitney": mann_p,
        }
    )

associations = pd.DataFrame(association_rows)
associations["p_ponto_bisserial_bh"] = benjamini_hochberg(associations["p_ponto_bisserial"])
associations["p_mann_whitney_bh"] = benjamini_hochberg(associations["p_mann_whitney"])
associations["abs_r_ponto_bisserial"] = associations["r_ponto_bisserial"].abs()

display(associations.sort_values("abs_r_ponto_bisserial", ascending=False).head(15))

### 3.14 PCA exploratória em nível de participante

In [ ]:
pca_input = subject_data[acoustic_columns].replace([np.inf, -np.inf], np.nan)
complete_columns = pca_input.columns[pca_input.notna().all()].tolist()
nonconstant_columns = [column for column in complete_columns if pca_input[column].var() > 0]

scaled_features = StandardScaler().fit_transform(pca_input[nonconstant_columns])
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coordinates = pca.fit_transform(scaled_features)

pca_frame = pd.DataFrame(pca_coordinates, columns=["PC1", "PC2"])
pca_frame[STATUS_COLUMN] = subject_data[STATUS_COLUMN].to_numpy()

figure, axis = plt.subplots(figsize=(8, 6))
for status_value, label, color in [(0, "Controle", "#4C78A8"), (1, "Parkinson", "#E45756")]:
    subset = pca_frame[pca_frame[STATUS_COLUMN] == status_value]
    axis.scatter(subset["PC1"], subset["PC2"], label=label, color=color, alpha=0.8)

axis.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% da variância)")
axis.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% da variância)")
axis.set_title("PCA das características agregadas por participante")
axis.legend()
figure.tight_layout()
plt.show()

## 4. Verificações e síntese

### 4.1 Resumo automático para apoiar a interpretação

In [ ]:
print("Participantes:", subject_data[ID_COLUMN].nunique())
print("Gravações:", len(data))
print("Ausências:", int(data.isna().sum().sum()))
print("Características acústicas numéricas:", len(acoustic_columns))
print("Variância explicada por PC1 + PC2:", f"{pca.explained_variance_ratio_[:2].sum() * 100:.1f}%")

print()
print("Cinco maiores associações ponto-bisseriais absolutas (exploratórias):")
display(
    associations.sort_values("abs_r_ponto_bisserial", ascending=False)[
        ["variavel", "r_ponto_bisserial", "cohens_d", "p_ponto_bisserial_bh"]
    ].head(5)
)

### 4.2 Principais observações — preencher após executar

Esta versão não contém conclusões numéricas porque o dataset não foi obtido durante a configuração. Após a execução, registre aqui apenas observações diretamente sustentadas pelas tabelas e figuras acima:

1. integridade estrutural, ausências e duplicatas;
2. confirmação de 80 participantes e três gravações por pessoa;
3. características com diferenças visuais/efeitos mais destacados, sem linguagem causal;
4. correlações altas e possível redundância;
5. estrutura observada na PCA e variância explicada, sem alegar separabilidade clínica.

### 4.3 Limitações da interpretação

- amostra pequena, regional e balanceada por desenho;
- três réplicas não independentes por participante;
- participantes com mais de 50 anos e composição por sexo potencialmente diferente entre grupos;
- apenas vogal sustentada `/a/` e protocolo específico de gravação;
- características já extraídas, sem áudio bruto para repetir o processamento;
- múltiplas comparações e caráter exploratório;
- PCA é descritiva e não mede desempenho diagnóstico;
- ausência de validação externa impede generalização clínica.

## 5. Próximos passos

1. executar o notebook e revisar os resultados produzidos;
2. registrar hash SHA-256 e data de obtenção do CSV, se houver cópia local;
3. investigar pressupostos e estabilidade das associações em nível de participante;
4. se a disciplina avançar para ML, criar pipeline com padronização e seleção dentro de cada dobra;
5. usar separação por `ID` (`GroupKFold`, `StratifiedGroupKFold` ou equivalente) e relatar sensibilidade, especificidade, balanced accuracy, F1 e ROC-AUC;
6. buscar validação externa antes de qualquer interpretação clínica.

### 5.1 Gate de agrupamento para ML futuro (sem treinar modelo)

In [ ]:
groups_for_future_ml = data[ID_COLUMN]
target_for_future_ml = data[STATUS_COLUMN]

assert groups_for_future_ml.nunique() == subject_data[ID_COLUMN].nunique()
print(
    "Qualquer divisão futura deve manter todas as linhas de um ID na mesma dobra. "
    "Não use train_test_split aleatório por linha."
)

## Fontes

- [UCI 489 — Parkinson Dataset with replicated acoustic features](https://archive.ics.uci.edu/dataset/489/parkinson+dataset+with+replicated+acoustic+features)
- [DOI do dataset: 10.24432/C5701F](https://doi.org/10.24432/C5701F)
- [Naranjo et al. (2016): desenho com réplicas](https://doi.org/10.1016/j.eswa.2015.10.034)
- [Naranjo et al. (2017): seleção e classificação em duas etapas](https://doi.org/10.1016/j.cmpb.2017.02.019)